# Module 1, Video 4: Vector Store Comparison
## pgvector vs OpenSearch Serverless — side-by-side

**What you'll do:**
- Generate embeddings for 100 products using Amazon Titan Embed v2
- Store them in Aurora PostgreSQL (pgvector)
- Store the same vectors in OpenSearch Serverless
- Query both with the same natural language search
- Compare results and understand why they differ

**Prerequisites:**
- `make deploy-base` and `make deploy-web` completed
- `make seed-data` completed (products in Aurora and OpenSearch)
- Amazon Bedrock model access enabled for Titan Embeddings V2

**Estimated cost:** ~£0.01 (100 embedding calls to Titan Embed v2)

## Setup

In [1]:
import boto3
import json
import time
from botocore.exceptions import ClientError

# Configuration
AWS_REGION = "eu-west-1"       # Change if you deployed to a different region
AWS_PROFILE = "ridge-course-dev"  # Your AWS CLI profile name
STACK_NAME = "meridian-base"   # CloudFormation stack name

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client("bedrock-runtime")
rds_data = session.client("rds-data")

# Auto-discover all ARNs from CloudFormation stack outputs
cfn = session.client("cloudformation")
outputs = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
stack_out = {o["OutputKey"]: o["OutputValue"] for o in outputs}

CLUSTER_ARN = stack_out["AuroraClusterArn"]
SECRET_ARN = stack_out["AuroraSecretArn"]
OPENSEARCH_ENDPOINT = stack_out.get("OpenSearchCollectionEndpoint", "")
DATABASE = "meridian"

print(f"Region:     {AWS_REGION}")
print(f"Aurora:     {CLUSTER_ARN.split(':cluster:')[1]}")
print(f"Secret:     ...{SECRET_ARN[-12:]}")
print(f"OpenSearch: {OPENSEARCH_ENDPOINT[:60]}" if OPENSEARCH_ENDPOINT else "OpenSearch: NOT FOUND")


Region:     eu-west-1
Aurora:     meridian-base-auroracluster-ybtfvjqlngjr
Secret:     ...a/dev-uOY21x
OpenSearch: https://ylbjh3ualyui79xwz3qd.aoss.eu-west-1.on.aws


## Step 1: Select sample products from Aurora

We'll pick 100 products with good descriptions across different categories.

In [2]:
def execute_sql(sql, params=None):
    """Execute SQL via RDS Data API with auto-pause retry."""
    kwargs = {
        "resourceArn": CLUSTER_ARN,
        "secretArn": SECRET_ARN,
        "database": DATABASE,
        "sql": sql,
        "includeResultMetadata": True,
    }
    if params:
        kwargs["parameters"] = params
    for attempt in range(4):
        try:
            resp = rds_data.execute_statement(**kwargs)
            return resp
        except ClientError as exc:
            if "DatabaseResumingException" in str(exc) and attempt < 3:
                print(f"  Aurora resuming... waiting {3*(attempt+1)}s")
                time.sleep(3 * (attempt + 1))
                continue
            raise

# Fetch 100 products with descriptions, spread across categories
resp = execute_sql("""
    SELECT sku, name, l1, brand, short_description
    FROM products
    WHERE short_description IS NOT NULL
      AND length(short_description) > 50
    ORDER BY random()
    LIMIT 100
""")

columns = [col["name"] for col in resp["columnMetadata"]]
products = []
for row in resp["records"]:
    product = {}
    for i, col in enumerate(columns):
        field = row[i]
        if "stringValue" in field:
            product[col] = field["stringValue"]
        elif "isNull" in field:
            product[col] = None
        else:
            product[col] = str(field)
    products.append(product)

print(f"Selected {len(products)} products:")
for p in products[:5]:
    print(f"  {p['sku']} — {p['name']} ({p['l1']})")
print(f"  ... and {len(products)-5} more")

Selected 100 products:
  MER-FMT-BSC-0039 — Forth Bridge Explorer 5 (Tents & Shelters)
  MER-SDL-TRH-0141 — Durham Ultra Light (Footwear)
  MER-CKW-MER-0021 — Loch Lomond Camping Pot (Camping & Cooking)
  MER-TVB-NOR-0017 — Cairngorm Adventure Pack (Rucksacks & Bags)
  MER-SLM-SMD-0037 — Stirling Ultralight (Sleeping)
  ... and 95 more


## Step 2: Generate embeddings with Titan Embed v2

Each product's short description becomes a 1024-dimensional vector.

In [3]:
def generate_embedding(text):
    """Call Titan Embed v2 to get a 1024-dim embedding."""
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,
            "normalize": True
        })
    )
    result = json.loads(response["body"].read())
    return result["embedding"]

# Generate embeddings for all products
print("Generating embeddings...")
for i, p in enumerate(products):
    text = f"{p['name']}. {p['short_description']}"
    p["embedding"] = generate_embedding(text)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(products)} done")

print(f"\nAll {len(products)} embeddings generated.")
print(f"Vector dimensions: {len(products[0]['embedding'])}")
print(f"Sample values: [{products[0]['embedding'][0]:.4f}, {products[0]['embedding'][1]:.4f}, {products[0]['embedding'][2]:.4f}, ...]")

Generating embeddings...
  5/100 done
  10/100 done
  15/100 done
  20/100 done
  25/100 done
  30/100 done
  35/100 done
  40/100 done
  45/100 done
  50/100 done
  55/100 done
  60/100 done
  65/100 done
  70/100 done
  75/100 done
  80/100 done
  85/100 done
  90/100 done
  95/100 done
  100/100 done

All 100 embeddings generated.
Vector dimensions: 1024
Sample values: [0.0076, 0.0015, 0.0260, ...]


## Step 3: Store in Aurora pgvector

We update the existing `embedding` column on the products table.

In [4]:
# Ensure pgvector extension exists
execute_sql("CREATE EXTENSION IF NOT EXISTS vector")

# Store embeddings in the products table
stored = 0
for p in products:
    vec_str = "[" + ",".join(str(v) for v in p["embedding"]) + "]"
    execute_sql(
        "UPDATE products SET embedding = :vec::vector WHERE sku = :sku",
        params=[
            {"name": "vec", "value": {"stringValue": vec_str}},
            {"name": "sku", "value": {"stringValue": p["sku"]}},
        ]
    )
    stored += 1

print(f"Stored {stored} embeddings in Aurora pgvector")

Stored 100 embeddings in Aurora pgvector


## Step 4: Store in OpenSearch Serverless

We index the same vectors into the existing OpenSearch `products` index, adding a `description_embedding` field.

In [5]:
import hashlib
import urllib.request
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

def opensearch_request(method, path, body=None):
    """Make a SigV4-signed request to OpenSearch Serverless."""
    url = f"{OPENSEARCH_ENDPOINT}{path}"
    data = json.dumps(body).encode() if body else b""
    content_hash = hashlib.sha256(data).hexdigest()
    headers = {
        "Content-Type": "application/json",
        "x-amz-content-sha256": content_hash,
    }
    credentials = session.get_credentials().get_frozen_credentials()
    request = AWSRequest(method=method, url=url, data=data, headers=headers)
    SigV4Auth(credentials, "aoss", AWS_REGION).add_auth(request)
    req = urllib.request.Request(url=request.url, data=data, headers=dict(request.headers), method=method)
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# Create a dedicated demo index with knn enabled
DEMO_INDEX = "products-vectors-demo"

# Delete if it exists from a previous run
try:
    opensearch_request("DELETE", f"/{DEMO_INDEX}")
    time.sleep(2)
except Exception:
    pass

# Create with knn enabled and the vector field mapped
index_body = {
    "settings": {"index": {"knn": True}},
    "mappings": {
        "properties": {
            "sku": {"type": "keyword"},
            "name": {"type": "text", "analyzer": "english"},
            "l1": {"type": "keyword"},
            "brand": {"type": "keyword"},
            "short_description": {"type": "text", "analyzer": "english"},
            "embedding_vector": {"type": "knn_vector", "dimension": 1024}
        }
    }
}
opensearch_request("PUT", f"/{DEMO_INDEX}", index_body)
print(f"Created index '{DEMO_INDEX}' with knn_vector mapping")

# Index each product with its embedding
indexed = 0
for p in products:
    doc = {
        "sku": p["sku"],
        "name": p["name"],
        "l1": p["l1"],
        "brand": p["brand"],
        "short_description": p["short_description"],
        "embedding_vector": p["embedding"]
    }
    try:
        opensearch_request("PUT", f"/{DEMO_INDEX}/_doc/{p['sku']}", doc)
        indexed += 1
    except Exception as e:
        print(f"  Failed: {p['sku']}: {e}")

print(f"Indexed {indexed} products with embeddings in OpenSearch")
print("Waiting for ANN index to build...")
time.sleep(5)


  Failed to update MER-WST-CPR-0040: HTTP Error 404: Not Found
  Failed to update MER-STV-WLK-0052: HTTP Error 404: Not Found
Indexed 98 embeddings in OpenSearch


## Step 5: Query both stores with the same search

Let's search for "warm waterproof jacket for hiking in Scotland" and compare results.

In [6]:
QUERY = "warm waterproof jacket for hiking in Scotland"

# Generate the query embedding
query_embedding = generate_embedding(QUERY)
print(f"Query: '{QUERY}'")
print(f"Query vector: [{query_embedding[0]:.4f}, {query_embedding[1]:.4f}, ...] (1024 dims)")

Query: 'warm waterproof jacket for hiking in Scotland'
Query vector: [-0.0528, -0.0381, ...] (1024 dims)


### BM25 keyword search (what the website uses today)

This is the baseline — the same search Ridge Assist currently runs. Pure keyword matching, no vectors.

In [ ]:
# Query the main products index with BM25 only (no vectors)
bm25_body = {
    "query": {
        "multi_match": {
            "query": QUERY,
            "fields": ["name^3", "brand^2", "short_description", "long_description"],
            "type": "best_fields"
        }
    },
    "size": 5,
    "_source": ["sku", "name", "l1", "brand"]
}

bm25_start = time.time()
bm25_result = opensearch_request("POST", "/products/_search", bm25_body)
bm25_time = (time.time() - bm25_start) * 1000
bm25_hits = bm25_result.get("hits", {}).get("hits", [])

bm25_results = [{**h["_source"], "score": h["_score"]} for h in bm25_hits]

print(f"BM25 keyword results ({bm25_time:.0f}ms):")
print(f"{'Rank':<5} {'SKU':<20} {'Name':<40} {'Score':<10}")
print("-" * 75)
for i, r in enumerate(bm25_results, 1):
    print(f"{i:<5} {r['sku']:<20} {r['name'][:38]:<40} {r.get('score', 0):.4f}")


### pgvector query (pure cosine similarity)

In [7]:
# Query pgvector — find nearest neighbours by cosine distance
vec_str = "[" + ",".join(str(v) for v in query_embedding) + "]"
pg_start = time.time()
resp = execute_sql(
    """SELECT sku, name, l1, brand,
              1 - (embedding <=> :query::vector) as similarity
       FROM products
       WHERE embedding IS NOT NULL
       ORDER BY embedding <=> :query::vector
       LIMIT 5""",
    params=[{"name": "query", "value": {"stringValue": vec_str}}]
)
pg_time = (time.time() - pg_start) * 1000

columns = [col["name"] for col in resp["columnMetadata"]]
pg_results = []
for row in resp["records"]:
    r = {}
    for i, col in enumerate(columns):
        field = row[i]
        if "stringValue" in field:
            r[col] = field["stringValue"]
        elif "doubleValue" in field:
            r[col] = field["doubleValue"]
        elif "isNull" in field:
            r[col] = None
    pg_results.append(r)

print(f"pgvector results ({pg_time:.0f}ms):")
print(f"{'Rank':<5} {'SKU':<20} {'Name':<40} {'Similarity':<10}")
print("-" * 75)
for i, r in enumerate(pg_results, 1):
    sim = f"{r.get('similarity', 0):.4f}" if r.get('similarity') else "N/A"
    print(f"{i:<5} {r['sku']:<20} {r['name'][:38]:<40} {sim}")

pgvector results (380ms):
Rank  SKU                  Name                                     Similarity
---------------------------------------------------------------------------
1     MER-SSJ-STO-0081     St Andrews Lightweight Jacket            0.6181
2     MER-JKT-STO-0129     Cairngorm 500 Jacket                     0.5189
3     MER-JKT-RES-0078     Lednock Hill Waterproof Jacket           0.4822
4     MER-JKT-MER-0011     Snowdonia Windproof Jacket               0.4699
5     MER-WPJ-RES-0063     Brecon Beacon Trail Jacket               0.4686


### OpenSearch query (hybrid: BM25 + vector)

In [8]:
# Query OpenSearch — knn vector similarity on the demo index
os_body = {
    "query": {
        "knn": {
            "embedding_vector": {
                "vector": query_embedding,
                "k": 5
            }
        }
    },
    "size": 5,
    "_source": ["sku", "name", "l1", "brand"]
}

os_start = time.time()
result = opensearch_request("POST", f"/{DEMO_INDEX}/_search", os_body)
os_time = (time.time() - os_start) * 1000
hits = result.get("hits", {}).get("hits", [])

os_results = [{**h["_source"], "score": h["_score"]} for h in hits]

print(f"\nOpenSearch knn results ({os_time:.0f}ms):")
print(f"{'Rank':<5} {'SKU':<20} {'Name':<40} {'Score':<10}")
print("-" * 75)
for i, r in enumerate(os_results, 1):
    print(f"{i:<5} {r['sku']:<20} {r['name'][:38]:<40} {r.get('score', 0):.4f}")


NameError: name 'DEMO_INDEX' is not defined

## Step 6: Compare results side-by-side

In [ ]:
print(f"\n{'='*90}")
print(f"COMPARISON: '{QUERY}'")
print(f"{'='*90}")
print(f"\n{'BM25 (keyword only)':<30} {'pgvector (cosine sim)':<30} {'OpenSearch knn (vector)':<30}")
print(f"{'Latency: ' + f'{bm25_time:.0f}ms':<30} {'Latency: ' + f'{pg_time:.0f}ms':<30} {'Latency: ' + f'{os_time:.0f}ms':<30}")
print(f"{'-'*30} {'-'*30} {'-'*30}")

max_rows = max(len(bm25_results), len(pg_results), len(os_results))
for i in range(max_rows):
    bm = bm25_results[i]['name'][:28] if i < len(bm25_results) else ''
    pg = pg_results[i]['name'][:28] if i < len(pg_results) else ''
    os_name = os_results[i]['name'][:28] if i < len(os_results) else ''
    print(f"{i+1}. {bm:<28} {i+1}. {pg:<28} {i+1}. {os_name}")

print(f"\n{'='*90}")
print("Key insights:")
print("  BM25: matches exact keywords only. Misses semantically similar products.")
print("  pgvector: finds meaning-similar products regardless of exact words used.")
print("  OpenSearch knn: same vector approach, different scoring normalisation.")
print(f"{'='*90}")


## What you've built

- Generated real embeddings using Amazon Titan Embed v2
- Stored vectors in two different stores (pgvector and OpenSearch)
- Queried both with the same natural language input
- Observed how scoring algorithms produce different rankings

**Key takeaway:** The vector store you choose affects not just cost and operations, but *which results your users see*. OpenSearch's hybrid approach finds products matching both keywords and meaning. pgvector's pure cosine similarity finds the semantically closest matches regardless of exact term overlap.

## Optional extensions

1. Try more queries: "lightweight tent for summer festivals", "birthday gift for a climber under £50"
2. Increase the sample to 100 or 500 products and observe latency changes
3. Compare results with 256-dim vs 1024-dim embeddings (change the `dimensions` parameter)